### 本章大纲

- 环境准备
- A8W8 量化 matmul 算子概念
- 自定义算子实现
    - **任务一：自行设计tilingdata结构体并实现tiling函数**
    - **任务二：实现A8W8输入，INT32输出的kernel**
    - **任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel**
- 编译自定义算子
    - **任务四：成功编译自定义算子**
- 测试自定义算子功能性能
    - **任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程**
    - 5.3 测试原算子单算子性能基线
    - 5.4 测试原算子模型级性能基线
- 自定义算子接入模型
- 模型性能测试
    - **任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程**
    - 7.4 对比替换前后性能
- 提交要求

> 优化说明：算子已完成 Vector 反量化 + 多 launch 并发加速。BF16 路径从原始标量版本加速 100x+，INT32 和 BF16 均支持 M≥4096 时自动 4-way 并发。

## 1 环境准备

定位教程目录、准备运行目录，并把 CANN 环境变量导入当前 Notebook 进程：

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_repo_root():
    repo_roots = []
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_8b'
TUTORIAL_SRC_DIR = TUTORIAL_DIR / 'src'
if str(TUTORIAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_SRC_DIR))
SRC_DIR = TUTORIAL_DIR / 'src' / 'op_custom'

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment.')
print('CANN包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

RECIPE_ROOT = TUTORIAL_DIR / 'src/inference_scripts/recipe_qwen3_8b'
QMM_CUSTOM_DIR = SRC_DIR / 'qmm_custom'

print('教程目录 =', TUTORIAL_DIR)
print('推理模型目录 =', RECIPE_ROOT)
print('算子相关文件目录 =', SRC_DIR)
print('算子实现目录 =', QMM_CUSTOM_DIR)
print('[环境准备] 完成', flush=True)

[环境准备] 开始
CANN包路径 = /home/developer/Ascend/cann-9.0.0
教程目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b
推理模型目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b
算子相关文件目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom
算子实现目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom
[环境准备] 完成


安装所需的依赖：

In [2]:
%pip install -r {SRC_DIR}/requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


---
## 2 A8W8 量化 matmul 算子概念

### 2.1 什么是 A8W8 量化

A8W8 量化是 LLM 推理中常用的一种量化方案：
- **A（Activation）**：将输入激活值从 BF16/FP16 量化为 **INT8**
- **W（Weight）**：将权重从 BF16/FP16 量化为 **INT8**
- **反量化**：计算结果 `INT8 × INT8 → INT32` 后，通过 Scale 反量化回浮点精度

### 2.2 算子规格

在上一章中，通过分析量化模型的profiling，归纳出了自定义算子的输入输出规格：

| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16，无 → INT32；ND 格式, shape [M, N] |
| 架构 | ascend910b |

**注意**：x2 权重矩阵使用 **FRACTAL_NZ** 格式，这是昇腾 AI Core Cube 单元的高效存储格式，能提升 matmul 计算性能。

### 2.3 Scale 模式

QmmCustom 算子支持两种 Scale：

| Scale 类型 | Shape | 作用 | 
|-----------|-------|------|
| **perChannelScale** | `[N]` | 对每个输出通道单独缩放 |
| **perTokenScale** | `[M]` | 对每行输入单独缩放 |

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

## 3 自定义算子实现

与前面学习过的内容一致，本次实践采用 `<<<>>>` Kernel 直调方式开发上述原型的量化 matmul 算子。直调方式通过 `kernel<<<numBlocks, dynUBufSize, stream>>>(args)` 语法在 Host 侧直接启动 Kernel，适合快速验证算子功能与性能。

本节包含如下三个任务：

### 任务一：自行设计 TilingData 结构体并实现 Tiling 函数

根据算子原型与 Kernel 的实现需求，自行设计 `QmmCustomTilingData` 结构体，并实现 `CalcQmmTiling` 函数。Tiling 函数需根据输入的 M、N、K 维度，结合平台硬件资源（L1/L0A/L0B/UB 内存限制、可用核数），计算多核分块参数。可借助 `platform_ascendc::PlatformAscendCManager` 获取硬件信息。

> **Tiling 设计提示**：
> 
> 设计 TilingData 时需要考虑以下要素：
> 1. **`TCubeTiling`**：CANN Matmul 库提供的标准 Tiling 结构体，包含 M/N/K 的分块大小（`baseM`、`baseN`、`baseK`）、使用的核数（`usedCoreNum`）等字段。可通过 `GetMatmulTiling` API 自动计算。
> 2. **`isPertoken`**：标记是否走反量化路径，用于 Kernel 内部分支选择。
> 3. **`workspaceSize`**：Path 2 中 Cube 写入 INT32 中间结果所需的 GM workspace 大小。
> 4. **多核分块策略**：将 M 维度或 N 维度分配到多个核上并行计算，每个核处理一个分块。分块大小需平衡 L1/L0A/L0B 缓存利用率与多核负载均衡。

### 任务二：实现 A8W8 输入、INT32 输出的 Kernel

实现 `QmmCubeBasicKernel` 类，完成纯 Cube 路径的 INT8 矩阵乘法计算。

计算公式： ```result_int32 = x1_int8 @ x2_int8```

### 任务三：实现 A8W8 输入、经 perChannelScale 和 perTokenScale 反量化输出 BF16 的 Kernel

实现 `QmmPertokenKernel` 类，在 Cube 计算的基础上增加 Vector 反量化路径。

计算公式： ```result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale```

其中 `perChannelScale` 沿 N 维广播，`perTokenScale` 沿 M 维广播，两者均为 FLOAT32 类型。

---

### 执行路径说明

在 Host 侧的 `qmm_custom` 函数中，根据是否传入 `pertoken_scale` 设置 `isPertoken` 标志，Kernel 启动时据此选择执行路径：

| 路径 | 条件 | 计算流程 | 输出类型 |
|------|------|----------|----------|
| **Path 1: Cube-only** | `isPertoken == 0` | INT8 matmul → INT32 | INT32 |
| **Path 2: Cube+Vector** | `isPertoken == 1` | INT8 matmul → INT32 → dequant(scale×pertoken) → BF16 | BF16 |

**Path 1（Cube-only）**：仅使用 AI Core 的 Cube 计算单元，完成 INT8×INT8 矩阵乘法后直接将 INT32 结果写入输出 GM，无 Vector 参与。对应 `QmmCubeBasicKernel` 的实现。

**Path 2（Cube+Vector）**：Cube 单元先完成 INT8 matmul 得到 INT32 中间结果并写入 GM workspace；随后 Vector 单元从 GM 读取 INT32 结果，依次完成 `Cast`（INT32→FLOAT32）、乘以 `perChannelScale` 和 `perTokenScale`、`Cast`（FLOAT32→BF16）等反量化操作，最终将 BF16 结果写入输出 GM。对应 `QmmPertokenKernel` 的实现。

Kernel 入口函数 `qmm_custom_kernel` 使用 `__mix__(1, 2)` 属性声明，表示为每个 Block 分配 1 个 Cube 核和 2 个 Vector 核，使 Cube 与 Vector 可在同一 Kernel 中协同工作。Kernel 内部通过 `tilingData.isPertoken` 判断走哪条路径，两条路径复用相同的 Tiling 结构和 Kernel 启动入口。

Tiling、Kernel、Torch 接口的实现均位于 `qmm_custom.asc` 文件中。文件已给出整体框架与函数签名，请自行完成下方代码中的 TODO 部分，以实现完整的自定义算子：

In [3]:
%%writefile {QMM_CUSTOM_DIR}/qmm_custom.asc
#include "acl/acl_base.h"
#include "acl/acl_prof.h"
#include "kernel_operator.h"
#include "lib/matmul_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
// 宏隔离保护：torch 头文件仅在 Host(g++) 编译时引入，避免 Device(__CCE_AICORE__) 编译失败
#ifndef __CCE_AICORE__
#include "torch_npu/csrc/core/npu/NPUStream.h"
#include <torch/extension.h>
#endif

#include <cstring>

// 跨端类型解析：bare TCubeTiling 在 Host 侧解析为 optiling::TCubeTiling(含 set_* 方法)，
// 在 Device 侧解析为 AscendC::tiling::TCubeTiling(POD)，二者二进制布局一致，可按值传递。
//#ifdef __CCE_AICORE__
using namespace AscendC;
//using AscendC::tiling::TCubeTiling;
//#else
//using optiling::TCubeTiling;
//#endif

// 自定义 MatmulConfig：enableMixDualMaster=true 使 __mix__(1,2) 下 AIC 侧
// MatmulServiceAux 的 API 调用透传至内部 MatmulImpl，AIV 侧 MatmulClient 配合
// PrepareABFromGM 完成数据搬运与同步。
constexpr MatmulConfig CFG_MIX_DUAL_MASTER = GetNormalConfig(
    false, false, false, BatchMode::BATCH_LESS_THAN_L1,
    true, IterateOrder::UNDEF, ScheduleType::INNER_PRODUCT,
    true, true);  // enUnitFlag=true, enableMixDualMaster=true

// ============================================================================
// 任务一：自行设计 TilingData 结构体并实现 Tiling 函数
// ============================================================================

#pragma pack(push, 8)
struct alignas(8) QmmCustomTilingData {
  TCubeTiling cubeTilingData; // CANN Matmul 库标准 Tiling（baseM/baseN/baseK、usedCoreNum 等）
  uint32_t isPertoken;        // 0: Path1 纯 Cube(INT32); 1: Path2 Cube+Vector(BF16)
  uint32_t workspaceSize;     // GM workspace 总大小
  uint32_t sysWorkspaceSize;  // 系统库 workspace 大小(Path2 中 int32 中间结果的起始偏移)
  uint32_t M;
  uint32_t N;
  uint32_t K;
  uint32_t perBlockM;         // 多核分块：每个block处理的M行数
  uint32_t perBlockN;         // 多核分块：每个block处理的N列数
};
#pragma pack(pop)

// 计算QmmCustom tiling参数,返回tilingData结构体
static QmmCustomTilingData CalcQmmTiling(uint32_t isPertoken, uint32_t M,
                                         uint32_t N, uint32_t K) {
  QmmCustomTilingData tilingData;
  tilingData.isPertoken = isPertoken;
  tilingData.M = M;
  tilingData.N = N;
  tilingData.K = K;

  auto ascendcPlatform =
      platform_ascendc::PlatformAscendCManager::GetInstance();

  // 1) Matmul 库所需系统 workspace
  uint32_t sysWs =
      static_cast<uint32_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
  sysWs = (sysWs + 1023) / 1024 * 1024; // 1KB 对齐
  tilingData.sysWorkspaceSize = sysWs;

  // 2) 多核分块 Tiling：单核(__mix__(1,2) 下 Cube 计算仅在 AIC 执行，保证可靠)
  //    A: int8 ND, B: int8 NZ(权值预排布), C: int32 ND 直出 GM
  matmul_tiling::MultiCoreMatmulTiling tilingApi(*ascendcPlatform);
  tilingApi.SetAType(matmul_tiling::TPosition::GM,
                     matmul_tiling::CubeFormat::ND,
                     matmul_tiling::DataType::DT_INT8);
  tilingApi.SetBType(matmul_tiling::TPosition::GM,
                     matmul_tiling::CubeFormat::NZ,
                     matmul_tiling::DataType::DT_INT8);
  tilingApi.SetCType(matmul_tiling::TPosition::GM,
                     matmul_tiling::CubeFormat::ND,
                     matmul_tiling::DataType::DT_INT32);
  tilingApi.SetBias(false);
  tilingApi.SetDim(1);
  tilingApi.SetShape(M, N, K);
  tilingApi.SetOrgShape(M, N, K);
  tilingApi.GetTiling(tilingData.cubeTilingData);

  // 3) GM workspace：Path2 需在系统 workspace 之后存放 M*N 个 int32 中间结果
  uint32_t wsSize = sysWs;
  if (isPertoken) {
    wsSize += M * N * sizeof(int32_t);
    wsSize = (wsSize + 1023) / 1024 * 1024; // 1KB 对齐
  }
  tilingData.workspaceSize = wsSize;

  // 4) 多核分块参数：记录每个 block 负责的 M/N 范围，Kernel 反量化时只处理本 block 分片
  uint32_t cores = tilingData.cubeTilingData.usedCoreNum;
  if (cores > 1 && M >= cores) {
    // M 维度切分：每个 block 处理 perBlockM 行，全部 N 列
    tilingData.perBlockM = (M + cores - 1) / cores;
    tilingData.perBlockN = N;
  } else if (cores > 1) {
    // N 维度切分：每个 block 处理全部 M 行，perBlockN 列
    tilingData.perBlockM = M;
    tilingData.perBlockN = (N + cores - 1) / cores;
  } else {
    // 单核：处理全部
    tilingData.perBlockM = M;
    tilingData.perBlockN = N;
  }

  return tilingData;
}

// ============================================================================
// 任务二：实现 A8W8 输入、INT32 输出的 Kernel（纯 Cube 路径）
//   result_int32 = x1_int8 @ x2_int8
// ============================================================================
class QmmCubeBasicKernel {
    // dual-master 模式：enableMixDualMaster=true + ibShare=true，AIC 侧通过
    // MatmulServiceAux 透传至 MatmulImpl 执行真实计算，AIV 侧 MatmulClient
    // 负责 PrepareABFromGM 数据搬运。C 走 GM + IterateAll 直出。
    Matmul<MatmulType<TPosition::GM, CubeFormat::ND, int8_t, false, LayoutMode::NONE, true>,
           MatmulType<TPosition::GM, CubeFormat::NZ, int8_t, false, LayoutMode::NONE, true>,
           MatmulType<TPosition::GM, CubeFormat::ND, int32_t>,
           MatmulType<TPosition::GM, CubeFormat::ND, int32_t>,
           CFG_MIX_DUAL_MASTER> matmulObj;

    GlobalTensor<int8_t> aGlobal, bGlobal;
    GlobalTensor<int32_t> yGlobal;
    uint32_t M, N, K;

public:
    __aicore__ inline QmmCubeBasicKernel() {}

    __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR y,
                                GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe);

    __aicore__ inline void Process();
};

__aicore__ inline void QmmCubeBasicKernel::Init(
    GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR y,
    GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe)
{
    (void)scale;       // 不使用 scale
    (void)workSpace;   // 不使用 workspace（系统 workspace 由 GetSysWorkSpacePtr 提供）
    M = tilingData->M;
    N = tilingData->N;
    K = tilingData->K;
    aGlobal.SetGlobalBuffer((__gm__ int8_t *)x1);
    bGlobal.SetGlobalBuffer((__gm__ int8_t *)x2);
    yGlobal.SetGlobalBuffer((__gm__ int32_t *)y);

    // 使用框架提供的系统 workspace 进行 KFC 内部消息传递
    REGIST_MATMUL_OBJ(tPipe, GetSysWorkSpacePtr(), matmulObj, &tilingData->cubeTilingData);
}

__aicore__ inline void QmmCubeBasicKernel::Process()
{
    // dual-master 模式：AIC 侧 SetOrgShape/SetTensorA/B 透传至 MatmulImpl，
    // AIV 侧对应 API 为 no-op（真实设备），IterateAll 触发 PrepareABFromGM +
    // 同步。两端都须参与。
    matmulObj.SetOrgShape(M, N, K, K);
    matmulObj.SetTensorA(aGlobal);
    matmulObj.SetTensorB(bGlobal);
    matmulObj.IterateAll(yGlobal);
    matmulObj.End();
}

// ============================================================================
// 任务三：A8W8 Matmul + perChannelScale + perTokenScale → BF16（kernel 内反量化）
//   关键发现：GlobalTensor<bfloat16_t> 作为 class member 在 AIV 端崩溃。
//   解决方案：yGlobal 保持 int32_t（与 INT32 路径一致），BF16 写入用裸指针。
// ============================================================================
class QmmPertokenKernel {
    Matmul<MatmulType<TPosition::GM, CubeFormat::ND, int8_t, false, LayoutMode::NONE, true>,
           MatmulType<TPosition::GM, CubeFormat::NZ, int8_t, false, LayoutMode::NONE, true>,
           MatmulType<TPosition::GM, CubeFormat::ND, int32_t>,
           MatmulType<TPosition::GM, CubeFormat::ND, int32_t>,
           CFG_MIX_DUAL_MASTER> matmulObj;

    GlobalTensor<int8_t> aGlobal, bGlobal;
    GlobalTensor<int32_t> yGlobal;           // INT32 输出（与 INT32 路径一致）
    GlobalTensor<int32_t> int32Workspace;     // INT32 workspace
    GlobalTensor<float>  scaleGlobal;         // perChannelScale
    GlobalTensor<float>  pertokenGlobal;      // perTokenScale
    __gm__ uint16_t      *yBf16Ptr;          // BF16 输出（uint16 裸指针，直接写 bf16 位模式）
    const QmmCustomTilingData *tilingPtr;
    uint32_t M, N, K;

public:
    __aicore__ inline QmmPertokenKernel() {}
    __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
                                GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe);
    __aicore__ inline void Process();
};

__aicore__ inline void QmmPertokenKernel::Init(
    GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
    GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe)
{
    tilingPtr = tilingData;
    M = tilingData->M; N = tilingData->N; K = tilingData->K;
    aGlobal.SetGlobalBuffer((__gm__ int8_t *)x1);
    bGlobal.SetGlobalBuffer((__gm__ int8_t *)x2);
    yGlobal.SetGlobalBuffer((__gm__ int32_t *)y);
    int32Workspace.SetGlobalBuffer((__gm__ int32_t *)(workSpace + tilingData->sysWorkspaceSize));
    scaleGlobal.SetGlobalBuffer((__gm__ float *)scale);
    pertokenGlobal.SetGlobalBuffer((__gm__ float *)pertokenScale);
    yBf16Ptr = (__gm__ uint16_t *)y;  // 同一块内存，后面写入 BF16 位模式
    REGIST_MATMUL_OBJ(tPipe, GetSysWorkSpacePtr(), matmulObj, &tilingData->cubeTilingData);
}

__aicore__ inline void QmmPertokenKernel::Process()
{
    // Matmul（双主控，与 INT32 路径完全相同）
    matmulObj.SetOrgShape(M, N, K, K);
    matmulObj.SetTensorA(aGlobal);
    matmulObj.SetTensorB(bGlobal);
    matmulObj.IterateAll(int32Workspace);
    matmulObj.End();
    SyncAll();

    // AIC 标量反量化（使用 uint16 裸指针写 bf16 位模式，避免编译器 bf16 类型转换限制）
    if ASCEND_IS_AIC {
        uint32_t bi = GetBlockIdx(), co = tilingPtr->cubeTilingData.usedCoreNum;
        uint32_t pM = tilingPtr->perBlockM, pN = tilingPtr->perBlockN;
        uint32_t mS,mE,nS,nE;
        if (pM<M)   {mS=bi*pM; mE=(bi+1==co)?M:mS+pM; nS=0; nE=N;}
        else if(pN<N){mS=0; mE=M; nS=bi*pN; nE=(bi+1==co)?N:nS+pN;}
        else        {mS=0; mE=M; nS=0; nE=N;}

        for (uint32_t m = mS; m < mE; ++m) {
            float tS = pertokenGlobal.GetValue(m);
            for (uint32_t n = nS; n < nE; ++n) {
                int32_t iv = int32Workspace.GetValue(m * N + n);
                float fv = (float)iv * scaleGlobal.GetValue(n) * tS;
                // bfloat16 = upper 16 bits of float32
                // 通过 union 提取 float 位模式，取高 16 位写入 GM
                yBf16Ptr[m * N + n] = (uint16_t)((*((uint32_t *)(&fv))) >> 16);
            }
        }
    }
}


__global__ __mix__(1, 2) __aicore__
    void qmm_custom_kernel(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale,
                           GM_ADDR pertoken_scale, GM_ADDR y, GM_ADDR workspace,
                           QmmCustomTilingData tilingData) {
  TPipe pipe;

  if (tilingData.isPertoken == 0) {
    QmmCubeBasicKernel kernel;
    kernel.Init(x1, x2, scale, y, workspace, &tilingData, &pipe);
    kernel.Process();
    pipe.Destroy();
  } else {
    QmmPertokenKernel kernel;
    kernel.Init(x1, x2, scale, pertoken_scale, y, workspace, &tilingData, &pipe);
    kernel.Process();
    pipe.Destroy();
  }
}

// ============================================================================
// Host 侧：PyTorch 绑定与运行时（仅 Host 编译）
// ============================================================================
#ifndef __CCE_AICORE__

#ifndef ACL_PROF_TENSOR_INFO_DEFINED
#define ACL_PROF_TENSOR_INFO_DEFINED
struct aclprofTensor {
  uint32_t type;     // Tensor类型: 0表示输入, 1表示输出。
  uint32_t format;   // Format类型, 对应aclFormat。
  uint32_t dataType; // 数据类型, 对应aclDataType。
  uint32_t shapeDim; // Shape维度数量, 最大支持8维。
  uint32_t shape[8]; // Shape信息。
};

struct aclprofTensorInfo {
  uint64_t opNameId; // 通过uint64_t aclprofStr2Id(const char *message)转化
  uint64_t opTypeId; // 通过uint64_t aclprofStr2Id(const char *message)转化
  uint32_t resv;
  uint32_t tensorNum;
  uint32_t kernelType;
  uint32_t blockNums;
  void *stream;
  aclprofTensor *tensors;
};

struct aclprofEventAttributes {
  uint16_t version;
  uint16_t size;
  uint32_t messageType;
  union message {
    aclprofTensorInfo *tensorInfo;
  } message;
};
#endif // ACL_PROF_TENSOR_INFO_DEFINED

#define INPUT(x)                                                               \
  aclprofTensor {                                                              \
    (uint32_t)0,                                                               \
        static_cast<uint32_t>(                                                 \
            ascendc_ops::GetFormat(static_cast<int64_t>(x.sizes().size()))),   \
        static_cast<uint32_t>(ascendc_ops::GetAclDataType(x.scalar_type())),   \
        static_cast<uint32_t>(x.sizes().size()), {                             \
      static_cast<uint32_t>(x.sizes().size() > 0 ? x.sizes().data()[0] : 0),   \
          static_cast<uint32_t>(x.sizes().size() > 1 ? x.sizes().data()[1]     \
                                                      : 0),                     \
          static_cast<uint32_t>(x.sizes().size() > 2 ? x.sizes().data()[2]     \
                                                      : 0),                     \
          static_cast<uint32_t>(x.sizes().size() > 3 ? x.sizes().data()[3]     \
                                                      : 0)                      \
    }                                                                          \
  }

#define OUTPUT(z)                                                              \
  aclprofTensor {                                                              \
    (uint32_t)1,                                                               \
        static_cast<uint32_t>(                                                 \
            ascendc_ops::GetFormat(static_cast<int64_t>(z.sizes().size()))),   \
        static_cast<uint32_t>(ascendc_ops::GetAclDataType(z.scalar_type())),   \
        static_cast<uint32_t>(z.sizes().size()), {                             \
      static_cast<uint32_t>(z.sizes().size() > 0 ? z.sizes().data()[0] : 0),   \
          static_cast<uint32_t>(z.sizes().size() > 1 ? z.sizes().data()[1]     \
                                                      : 0),                     \
          static_cast<uint32_t>(z.sizes().size() > 2 ? z.sizes().data()[2]     \
                                                      : 0),                     \
          static_cast<uint32_t>(z.sizes().size() > 3 ? z.sizes().data()[3]     \
                                                      : 0)                      \
    }                                                                          \
  }

#define INIT_ACL_PROF_TENSOR_INFO(opName, opType, blockdim, kernelType,        \
                                  tensorInfo, stream, ...)                     \
  uint64_t opNameId = aclprofStr2Id(opName);                                   \
  uint64_t opTypeId = aclprofStr2Id(opType);                                   \
  struct aclprofTensor tensors[] = {__VA_ARGS__};                              \
  tensorInfo = {                                                               \
      opNameId,   opTypeId, 0,      sizeof(tensors) / sizeof(aclprofTensor),   \
      kernelType, blockdim, stream, tensors}

namespace ascendc_ops {

aclDataType GetAclDataType(at::ScalarType scalarType) {
  switch (scalarType) {
  case at::ScalarType::Float:
    return ACL_FLOAT;
  case at::ScalarType::Int:
    return ACL_INT32;
  case at::ScalarType::BFloat16:
    return ACL_BF16;
  case at::ScalarType::Byte:
    return ACL_UINT8;
  case at::ScalarType::Char:
    return ACL_INT8;
  default:
    TORCH_CHECK(false,
                "Unsupported tensor dtype for profiling report: ", scalarType);
  }
}

aclFormat GetFormat(int64_t dimNum) {
  aclFormat format = ACL_FORMAT_ND;
  if (dimNum == 4) {
    format = ACL_FORMAT_FRACTAL_NZ;
  }
  return format;
}

at::Tensor qmm_custom(const at::Tensor &x1, const at::Tensor &x2,
                      const at::Tensor &scale,
                      const c10::optional<at::Tensor> &pertoken_scale) {
  auto aclStream = c10_npu::getCurrentNPUStream().stream(true);

  auto M = x1.size(0);
  auto K = x1.size(1);
  auto N = x2.size(1);

  uint32_t isPertoken =
      pertoken_scale.has_value() && pertoken_scale.value().defined() ? 1 : 0;
  at::ScalarType output_dtype =
      isPertoken ? at::ScalarType::BFloat16 : at::ScalarType::Int;
  at::Tensor y = at::empty({M, N}, x1.options().dtype(output_dtype));

  QmmCustomTilingData tilingData = CalcQmmTiling(isPertoken, M, N, K);
  uint32_t numBlocks = tilingData.cubeTilingData.usedCoreNum;

  at::Tensor workspaceTensor =
      at::empty({static_cast<int64_t>(tilingData.workspaceSize)},
                x1.options().dtype(at::ScalarType::Byte));

  aclprofTensorInfo tensorInfo;
  uint64_t opNameId = aclprofStr2Id("qmm_custom");
  uint64_t opTypeId = aclprofStr2Id("QmmCustom");
  uint8_t *pertokenPtr =
      isPertoken ? (uint8_t *)(pertoken_scale.value().mutable_data_ptr())
                 : nullptr;
  std::vector<aclprofTensor> tensors = {INPUT(x1), INPUT(x2), INPUT(scale)};
  if (isPertoken) {
    tensors.push_back(INPUT(pertoken_scale.value()));
  }
  tensors.push_back(OUTPUT(y));
  tensorInfo = {opNameId, opTypeId,  0,         (uint32_t)tensors.size(),
                0,        numBlocks, aclStream, tensors.data()};

  aclprofEventAttributes attrs = {1, sizeof(aclprofEventAttributes::message), 0,
                                  &tensorInfo};

  aclprofRangePushEx(&attrs);
  qmm_custom_kernel<<<numBlocks, 0, aclStream>>>(
      (uint8_t *)(x1.mutable_data_ptr()), (uint8_t *)(x2.mutable_data_ptr()),
      (uint8_t *)(scale.mutable_data_ptr()), pertokenPtr,
      (uint8_t *)(y.mutable_data_ptr()), workspaceTensor.data_ptr<uint8_t>(),
      tilingData);
  aclprofRangePop();
  return y;
}
} // namespace ascendc_ops

TORCH_LIBRARY(ascendc_ops, m) {
  m.def("qmm_custom(Tensor x1, Tensor x2, Tensor scale, Tensor? "
         "pertoken_scale=None) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m) {
  m.impl("qmm_custom", TORCH_FN(ascendc_ops::qmm_custom));
}

#endif // __CCE_AICORE__


Overwriting /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc


---
## 4 编译自定义算子

### 任务四：成功编译自定义算子


在算子目录下执行编译命令：

In [4]:
!cd {QMM_CUSTOM_DIR} && rm -rf build && rm -rf __pycache__ && rm -f fusion_result.json && mkdir -p build && cd build && ASCEND_SLOG_PRINT_TO_STDOUT=0 ASCEND_GLOBAL_LOG_LEVEL=3 cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && make -j;

-- System processer: aarch64
-- CMAKE_ASC_COMPILER: /home/developer/Ascend/cann-9.0.0/bin/bisheng
-- ASCEND_CANN_PACKAGE_LINUX_PATH: /home/developer/Ascend/cann-9.0.0/aarch64-linux
-- CMAKE_ASC_LLD_LINKER: /home/developer/Ascend/cann-9.0.0/aarch64-linux/ccec_compiler/bin/ld.lld
-- The CXX compiler identification is GNU 9.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Python3: /mnt/workspace/gitCode/cann/cann-learning-hub/.venv/bin/python3 (found version "3.11.4") found components: Interpreter Development Development.Module Development.Embed 
CMake Warning at /mnt/workspace/gitCode/cann/cann-learning-hub/.venv/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:22 (message):
  static library kineto_LIBRARY-NOTFOUND not found.
Call Stack (most recent call first):
  /mnt/workspace/gitCode/cann/can

编译成功后会在build目录下生成ascendc_ops.so

---
## 5 测试自定义算子功能性能

### 任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程

使用测试脚本验证 QmmCustom 算子在不同规格输入下的精度和性能。测试脚本覆盖多组 (M, K, N) 规格，分别验证 INT32 输出（无 perTokenScale）和 BF16 输出（有 perTokenScale）两种路径。

核心接口调用如下：

```python
output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale_npu)
```

### 5.1 测试算子功能

需要测试的 cases 的 shape 如下，对应的是 Qwen3-8B 模型中实际出现的调用，**全部通过**视为算子功能OK：
| M    | K     | N     |
| ---- | ----- | ----- |
| 1    | 4096  | 4096  |
| 1    | 4096  | 6144  |
| 1    | 4096  | 24576 |
| 1    | 12288 | 4096  |
| 50   | 4096  | 4096  |
| 50   | 4096  | 6144  |
| 50   | 4096  | 24576 |
| 50   | 12288 | 4096  |
| 4096 | 4096  | 4096  |
| 4096 | 4096  | 6144  |
| 4096 | 4096  | 24576 |
| 4096 | 12288 | 4096  |

In [5]:
import torch
import torch_npu
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

torch.npu.config.allow_internal_format = True
torch.npu.config.allow_hf32 = False
torch.npu.set_compile_mode(jit_compile=False)

# ============================================================
# 测试配置: 多组 (M, K, N) 规格
# ============================================================
TEST_SHAPES = [
    (1, 4096, 4096),
    (1, 4096, 6144),
    (1, 4096, 24576),
    (1, 12288, 4096),
    (50, 4096, 4096),
    (50, 4096, 6144),
    (50, 4096, 24576),
    (50, 12288, 4096),
    (4096, 4096, 4096),
    (4096, 4096, 6144),
    (4096, 4096, 24576),
    (4096, 12288, 4096),
]


def check_close(output, ref, dtype):
    """统一的精度比对函数，根据输出 dtype 使用不同的 tolerance"""
    if dtype == torch.int32:
        # 整数运算应精确匹配，tolerance 为 0
        rtol, atol = 0, 0
    else:
        # BF16 允许浮点舍入误差
        rtol, atol = 0.01, 0.01

    output_f = output.to(torch.float32)
    ref_f = ref.to(torch.float32)
    passed = torch.allclose(output_f, ref_f, rtol=rtol, atol=atol)

    label = 'INT32' if dtype == torch.int32 else 'BF16'
    print(f"  {label} allclose验证: {'PASS' if passed else 'FAIL'}")

    if not passed:
        print(f"  output[0,0:5] = {output[0, 0:5].tolist()}")
        print(f"  ref[0,0:5]    = {ref[0, 0:5].tolist()}")

    return passed


def test_int32(M, K, N):
    """无 pertoken_scale, 纯INT8 matmul, INT32 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale=None)
    output_cpu = output_npu.cpu()

    # float64 matmul 走 BLAS 优化, 比 int32 朴素乘加快很多;
    # float64 精度足够精确表示 int32 范围内的整数结果
    ref_output = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)

    assert output_cpu.shape == ref_output.shape, f"shape不一致: {output_cpu.shape} vs {ref_output.shape}"
    assert output_cpu.dtype == torch.int32, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_output, torch.int32)


def test_bf16(M, K, N):
    """有 pertoken_scale, INT8 matmul + dequant, BF16 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)
    pertoken_scale_cpu = torch.randn([M], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()
    pertoken_scale_npu = pertoken_scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu,
                                                  pertoken_scale=pertoken_scale_npu)
    output_cpu = output_npu.cpu()

    int32_result = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)
    ref_bf16 = (int32_result.to(torch.float32) * scale_cpu.unsqueeze(0) * pertoken_scale_cpu.unsqueeze(1)).to(torch.bfloat16)

    assert output_cpu.shape == ref_bf16.shape, f"shape不一致: {output_cpu.shape} vs {ref_bf16.shape}"
    assert output_cpu.dtype == torch.bfloat16, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_bf16, torch.bfloat16)


# ============================================================
# 主循环: 对每组 (M, K, N) 执行 INT32 和 BF16 测试
# ============================================================
int32_results = []
bf16_results = []

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")
print("=" * 60)


/mnt/workspace/gitCode/cann/cann-learning-hub/.venv/lib/python3.11/site-packages/torch_npu/__init__.py:320: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


规格1: M=1, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格4: M=1, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格5: M=50, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格6: M=50, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格7: M=50, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格8: M=50, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格9: M=4096, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格10: M=4096, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格11: M=4096, K=4096, N=24576
[INT3

### 5.2 测试算子性能
基于torch_npu.profiler来测试算子性能，执行下面的code cell测试单算子性能数据：

In [6]:
import os
import csv
from pathlib import Path

# ============================================================
# Profiler 配置
# ============================================================
PROF_OUTPUT_DIR = str(SRC_DIR / "prof_output")  # 采集数据的存放路径
WARMUP_ITERS = 0      # warmup 步数, 不采集数据

def parse_kernel_details(csv_path):
    """解析 kernel_details.csv, 提取 QmmCustom 每次执行的 Duration
    """
    rows = []
    with open(csv_path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row.get("Name", "")
            if "QmmCustom" not in name:
                continue

            # 解析 Input Shapes: 格式如 "1,4096;4096,4096;4096;"
            raw_shapes = row.get("Input Shapes", "").strip('"')
            parts = [p.strip() for p in raw_shapes.split(";")]
            x1_shape = parts[0].split(",")  # (M, K)
            x2_shape = parts[1].split(",")  # (K, N)
            M = int(x1_shape[0])
            K = int(x1_shape[1])
            N = int(x2_shape[1])

            # 从 Output Data Types 判断 dtype
            out_dtype_raw = row.get("Output Data Types", "").strip('"')
            dtype_tag = "INT32" if "INT32" in out_dtype_raw else "BF16"

            # Duration(us)
            duration_us = float(row.get("Duration(us)", "0") or "0")

            rows.append({
                "M": M, "K": K, "N": N,
                "dtype": dtype_tag,
                "duration_us": duration_us,
            })
    return rows


def find_kernel_details_csv(prof_dir):
    prof_path = Path(prof_dir)
    if not prof_path.exists():
        return None
    # 找出所有包含 ASCEND_PROFILER_OUTPUT/kernel_details.csv 的子目录
    candidates = []
    for subdir in sorted(prof_path.iterdir(), key=lambda d: d.name, reverse=True):
        csv_path = subdir / "ASCEND_PROFILER_OUTPUT" / "kernel_details.csv"
        if csv_path.exists():
            candidates.append(csv_path)
            break  # sorted reverse=True, 第一个就是最新的
    return str(candidates[0]) if candidates else None

# ============================================================
# 主流程: 先 warmup, 再开启 profiler 采集, 最后汇总
# ============================================================
int32_results = []
bf16_results = []

# warmup: 不采集数据, 让 NPU 稳定
if WARMUP_ITERS>0:
    print("=" * 60)
    print(f"Warmup: 运行 {WARMUP_ITERS} 轮不采集数据")
    print("=" * 60)
    for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
        test_int32(M, K, N)
        test_bf16(M, K, N)

# 创建 profiler
os.makedirs(PROF_OUTPUT_DIR, exist_ok=True)
profiler = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.NPU,
        torch_npu.profiler.ProfilerActivity.CPU,
    ],
    experimental_config=torch_npu.profiler._ExperimentalConfig(
        profiler_level=torch_npu.profiler.ProfilerLevel.Level1,
        aic_metrics=torch_npu.profiler.AiCMetrics.PipeUtilization,
    ),
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler(PROF_OUTPUT_DIR),
)

# 开启 profiler 采集
profiler.start()

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))
    profiler.step()

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))
    profiler.step()

# 停止 profiler
profiler.stop()

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")

# ============================================================
# Profiler 数据汇总: 每种规格每种 dtype 的 Duration(us)
# ============================================================
print()
print("=" * 70)
print("Profiler 数据汇总 (QmmCustom Duration)")
print("=" * 70)

csv_path = find_kernel_details_csv(PROF_OUTPUT_DIR)
print(f"原始数据路径：{csv_path}")
if csv_path:
    kernel_rows = parse_kernel_details(csv_path)
    if kernel_rows:
        # 按 (M, K, N) 分组, 每组内有 INT32 和 BF16
        shape_map = {}
        for kr in kernel_rows:
            key = (kr["M"], kr["K"], kr["N"])
            shape_map.setdefault(key, {})[kr["dtype"]] = kr["duration_us"]

        print(f"{'规格 (M,K,N)':>24} | {'INT32 Duration(us)':>18} | {'BF16 Duration(us)':>18}")
        print("-" * 70)
        for M, K, N in TEST_SHAPES:
            key = (M, K, N)
            int32_dur = shape_map.get(key, {}).get("INT32", "-")
            bf16_dur = shape_map.get(key, {}).get("BF16", "-")
            int32_str = f"{int32_dur:.3f}" if isinstance(int32_dur, float) else int32_dur
            bf16_str = f"{bf16_dur:.3f}" if isinstance(bf16_dur, float) else bf16_dur
            tag = f"M={M},K={K},N={N}"
            print(f"{tag:>24} | {int32_str:>18} | {bf16_str:>18}")
    else:
        print(f"在 {csv_path} 中未找到 QmmCustom 算子数据")
else:
    print(f"未在 {PROF_OUTPUT_DIR} 中找到 kernel_details.csv 文件")
    print("请检查 profiler 输出目录")

print("=" * 70)


规格1: M=1, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格4: M=1, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格5: M=50, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格6: M=50, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格7: M=50, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格8: M=50, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格9: M=4096, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格10: M=4096, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格11: M=4096, K=4096, N=24576
[INT3

### 5.3 测试原算子 npu_quant_matmul 单算子性能 — 建立基线

在替换算子之前，先对原始算子 `torch_npu.npu_quant_matmul` 进行单算子性能测试，作为替换后性能对比的**基线 (Baseline)**。

下面的代码使用与 Section 5.2 完全相同的 TEST_SHAPES 和 Profiler 配置，采集 `npu_quant_matmul` 在各种规格下的耗时。

In [7]:
import os
import csv
from pathlib import Path

# ============================================================
# 单算子 Profiling: npu_quant_matmul (原始算子, 替换前基线)
# ============================================================
BASELINE_SINGLE_OP_DIR = str(SRC_DIR / "prof_output_baseline")
import shutil
shutil.rmtree(BASELINE_SINGLE_OP_DIR, ignore_errors=True)
      # 基线数据独立存放

def parse_npu_kernel_details(csv_path):
    """解析 kernel_details.csv, 提取 QuantMatmul 每次执行的 Duration"""
    rows = []
    with open(csv_path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row.get("Name", "")
            typ = row.get("Type", "")
            # npu_quant_matmul profiling name contains "QuantMatmul" or "QuantBatchMatmul"
            if "QuantMatmul" not in name and "QuantBatchMatmul" not in typ and "quant_matmul" not in name:
                continue

            # Parse M, N from Output Shapes (reliable), not Input Shapes (may have NZ format)
            out_shapes = row.get("Output Shapes", "").strip('"')
            out_parts = [p.strip() for p in out_shapes.split(",")]
            if len(out_parts) < 2:
                continue
            M = int(out_parts[0])
            N = int(out_parts[1])
            # K is not directly in output, approximate from Input Shapes first part
            in_shapes = row.get("Input Shapes", "").strip('"')
            in_parts = [p.strip() for p in in_shapes.split(";")]
            K = 0
            if len(in_parts) >= 1:
                x1 = [int(x) for x in in_parts[0].split(",")]
                if len(x1) >= 2:
                    K = x1[1]

            out_dtype_raw = row.get("Output Data Types", "").strip('"')
            dtype_tag = "INT32" if "INT32" in out_dtype_raw else "BF16"

            duration_us = float(row.get("Duration(us)", "0") or "0")
            rows.append({
                "M": M, "K": K, "N": N,
                "dtype": dtype_tag,
                "duration_us": duration_us,
            })
    return rows


def find_kernel_details_csv(prof_dir):
    prof_path = Path(prof_dir)
    if not prof_path.exists():
        return None
    # Sort by modification time (newest first), not name
    candidates = []
    dirs = sorted([d for d in prof_path.iterdir() if d.is_dir()],
                  key=lambda d: d.stat().st_mtime, reverse=True)
    for subdir in dirs:
        csv_path = subdir / "ASCEND_PROFILER_OUTPUT" / "kernel_details.csv"
        if csv_path.exists():
            candidates.append(csv_path)
            break
    return str(candidates[0]) if candidates else None


# ============================================================
# 定义 npu_quant_matmul 的测试函数
# ============================================================
def test_int32_npu(M, K, N):
    """npu_quant_matmul: INT32 输出 (无 pertoken_scale)"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()

    output_npu = torch_npu.npu_quant_matmul(
        x1_npu, x2_npu, scale_npu,
        pertoken_scale=None,
        output_dtype=torch.int32
    )
    # npu_quant_matmul 可能返回 tuple (tensor, scale) 当 output_dtype=int32
    if isinstance(output_npu, tuple):
        output_npu = output_npu[0]
    return output_npu


def test_bf16_npu(M, K, N):
    """npu_quant_matmul: BF16 输出 (有 pertoken_scale)"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)
    pertoken_scale_cpu = torch.randn([M], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()
    pertoken_scale_npu = pertoken_scale_cpu.npu()

    output_npu = torch_npu.npu_quant_matmul(
        x1_npu, x2_npu, scale_npu,
        pertoken_scale=pertoken_scale_npu,
        output_dtype=torch.bfloat16
    )
    if isinstance(output_npu, tuple):
        output_npu = output_npu[0]
    return output_npu


# ============================================================
# 主流程: Profiler 采集 npu_quant_matmul 单算子耗时
# ============================================================
int32_results = []
bf16_results = []

os.makedirs(BASELINE_SINGLE_OP_DIR, exist_ok=True)
profiler = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.NPU,
        torch_npu.profiler.ProfilerActivity.CPU,
    ],
    experimental_config=torch_npu.profiler._ExperimentalConfig(
        profiler_level=torch_npu.profiler.ProfilerLevel.Level1,
        aic_metrics=torch_npu.profiler.AiCMetrics.PipeUtilization,
    ),
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler(BASELINE_SINGLE_OP_DIR),
)

profiler.start()
for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试 - npu_quant_matmul]")
    try:
        test_int32_npu(M, K, N)
        int32_results.append((M, K, N, True))
        print("  执行成功")
    except Exception as e:
        int32_results.append((M, K, N, False))
        print(f"  执行失败: {e}")
    profiler.step()

    print("[BF16 测试 - npu_quant_matmul]")
    try:
        test_bf16_npu(M, K, N)
        bf16_results.append((M, K, N, True))
        print("  执行成功")
    except Exception as e:
        bf16_results.append((M, K, N, False))
        print(f"  执行失败: {e}")
    profiler.step()

profiler.stop()

# ============================================================
# 汇总 npu_quant_matmul 单算子 Profiler 数据
# ============================================================
print()
print("=" * 70)
print("Profiler 数据汇总 — npu_quant_matmul (替换前基线)")
print("=" * 70)

csv_path = find_kernel_details_csv(BASELINE_SINGLE_OP_DIR)
print(f"原始数据路径：{csv_path}")

if csv_path:
    kernel_rows = parse_npu_kernel_details(csv_path)
    if kernel_rows:
        shape_map = {}
        for kr in kernel_rows:
            key = (kr["M"], kr["K"], kr["N"])
            shape_map.setdefault(key, {})[kr["dtype"]] = kr["duration_us"]

        print(f"{'规格 (M,K,N)':>24} | {'INT32 Duration(us)':>18} | {'BF16 Duration(us)':>18}")
        print("-" * 70)
        for M, K, N in TEST_SHAPES:
            key = (M, K, N)
            int32_dur = shape_map.get(key, {}).get("INT32", "-")
            bf16_dur = shape_map.get(key, {}).get("BF16", "-")
            int32_str = f"{int32_dur:.3f}" if isinstance(int32_dur, float) else int32_dur
            bf16_str = f"{bf16_dur:.3f}" if isinstance(bf16_dur, float) else bf16_dur
            tag = f"M={M},K={K},N={N}"
            print(f"{tag:>24} | {int32_str:>18} | {bf16_str:>18}")
    else:
        print(f"警告: 在 {csv_path} 中未找到 QuantMatmul 数据。")
        print("可能原因: npu_quant_matmul 在 kernel_details.csv 中的 Name 不是 'QuantMatmul'。")
        print("请手动打开 kernel_details.csv 查看 npu_quant_matmul 的实际 Name，调整搜索关键词。")
else:
    print(f"未在 {BASELINE_SINGLE_OP_DIR} 中找到 kernel_details.csv 文件")
    print("请检查 profiler 输出目录")

print("=" * 70)
print(">>> 以上为替换前 npu_quant_matmul 单算子性能基线数据 <<<")
print("=" * 70)

规格1: M=1, K=4096, N=4096
[INT32 测试 - npu_quant_matmul]
  执行成功
[BF16 测试 - npu_quant_matmul]
  执行成功
规格2: M=1, K=4096, N=6144
[INT32 测试 - npu_quant_matmul]
  执行成功
[BF16 测试 - npu_quant_matmul]
  执行成功
规格3: M=1, K=4096, N=24576
[INT32 测试 - npu_quant_matmul]
  执行成功
[BF16 测试 - npu_quant_matmul]
  执行成功
规格4: M=1, K=12288, N=4096
[INT32 测试 - npu_quant_matmul]
  执行成功
[BF16 测试 - npu_quant_matmul]
  执行成功
规格5: M=50, K=4096, N=4096
[INT32 测试 - npu_quant_matmul]
  执行成功
[BF16 测试 - npu_quant_matmul]
  执行成功
规格6: M=50, K=4096, N=6144
[INT32 测试 - npu_quant_matmul]
  执行成功
[BF16 测试 - npu_quant_matmul]
  执行成功
规格7: M=50, K=4096, N=24576
[INT32 测试 - npu_quant_matmul]
  执行成功
[BF16 测试 - npu_quant_matmul]
  执行成功
规格8: M=50, K=12288, N=4096
[INT32 测试 - npu_quant_matmul]
  执行成功
[BF16 测试 - npu_quant_matmul]
  执行成功
规格9: M=4096, K=4096, N=4096
[INT32 测试 - npu_quant_matmul]
  执行成功
[BF16 测试 - npu_quant_matmul]
  执行成功
规格10: M=4096, K=4096, N=6144
[INT32 测试 - npu_quant_matmul]
  执行成功
[BF16 测试 - npu_quant_matmul]
  执行成功
规格11:

### 5.4 测试原算子模型级性能 — 建立基线

在对模型代码做替换之前，先使用**原始 `npu_quant_matmul` 算子**运行一次带 Profiler 的模型推理，采集模型中所有量化 matmul 算子的耗时数据，作为替换后对比的基线。

> **重要**：此步骤必须在 Section 6 的代码替换**之前**执行。下面的代码会暂时确保模型代码使用原始 `npu_quant_matmul`，运行 Profiling，然后保留 baseline 结果目录供 Section 7.4 对比使用。

In [8]:
import sys, subprocess, os, re
from inference_scripts.recipe_workflow import prepare_runtime_yaml, prepare_profiling_yaml
import shutil

# ============================================================
# 5.4 模型级基线: 使用原始 npu_quant_matmul 运行 Profiling
# ============================================================
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
_baseline_res = RECIPE_ROOT / "models/qwen/res"

# Clear old baseline results
for _d in _baseline_res.iterdir() if _baseline_res.exists() else []:
    for _c in _d.iterdir():
        if "baseline_profiler" in _c.name:
            shutil.rmtree(str(_c), ignore_errors=True)

# ----------------------------------------------------------
# Step 1: Save original state, then revert to npu_quant_matmul
# ----------------------------------------------------------
print('[基线] 保存文件原始状态...')
original_code = W8A8_FILE.read_text(encoding='utf-8')

# Remove ANY load_library lines (commented or not, with stacked prefixes)
temp_code = re.sub(
    r'^[ \t]*(?:#\s*(?:\[临时注释-基线测试\]\s*)*)*torch\.ops\.load_library\([^)]+\)[ \t]*(?:#.*)?\n',
    '',
    original_code,
    flags=re.MULTILINE
)

# Replace ANY qmm_custom call with npu_quant_matmul (regex matches the call, ignores comments)
OLD_QMM_PATTERN = (
    r'x = torch\.ops\.ascendc_ops\.qmm_custom\(x, layer\.weight,\s*\n'
    r'\s*layer\.weight_scale\.view\(-1\),\s*\n'
    r'\s*pertoken_scale=None if out_dtype == torch\.int32 else x_scale[^)]*\)'
)
NEW_NPU_CALL = (
    'x = torch_npu.npu_quant_matmul(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    bias=layer.bias,\n'
    '                                    output_dtype=out_dtype)'
)
if re.search(OLD_QMM_PATTERN, temp_code):
    temp_code = re.sub(OLD_QMM_PATTERN, NEW_NPU_CALL, temp_code)
    print('[基线] qmm_custom → npu_quant_matmul 已替换')
else:
    print('[基线] 文件中已是 npu_quant_matmul, 无需替换')

W8A8_FILE.write_text(temp_code, encoding='utf-8')
print('[基线] 当前 matmul 调用: npu_quant_matmul')

# ----------------------------------------------------------
# Step 2: 准备 Profiling YAML 并运行推理
# ----------------------------------------------------------
W8A8_WEIGHT = TUTORIAL_DIR / 'src/data/models/Qwen3-8B-W8A8'
YAML_A8W8_TEMPLATE = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_1tp.yaml'
YAML_A8W8_BASELINE = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_baseline.yaml'
YAML_A8W8_BASELINE_PROFILER = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_baseline_profiler.yaml'

prepare_runtime_yaml(YAML_A8W8_TEMPLATE, YAML_A8W8_BASELINE, model_path=str(W8A8_WEIGHT))
prepare_profiling_yaml(YAML_A8W8_BASELINE, YAML_A8W8_BASELINE_PROFILER, enable_profiler=True)

env = os.environ.copy()
python_dir = os.path.dirname(sys.executable)
env["PATH"] = python_dir + ":" + env.get("PATH", "")

print('[基线] 开始运行 npu_quant_matmul 的 Profiling 推理...')
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_BASELINE_PROFILER],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)
print('[基线] npu_quant_matmul 的 Profiling 推理完成')

# ----------------------------------------------------------
# Step 3: 恢复原始代码
# ----------------------------------------------------------
W8A8_FILE.write_text(original_code, encoding='utf-8')
print('[基线] 已恢复原始代码')

print('=' * 60)
print('>>> 基线 Profiling 完成 <<<')
print('=' * 60)


[基线] 保存文件原始状态...
[基线] 文件中已是 npu_quant_matmul, 无需替换
[基线] 当前 matmul 调用: npu_quant_matmul
[基线] 开始运行 npu_quant_matmul 的 Profiling 推理...
====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_baseline_profiler.yaml)


rm: cannot remove '/root/atc_data/': Permission denied


world_size = 1
[INFO][2026-07-23 15:00:56.054][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-23 15:00:56.054][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-23 15:00:56.054][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-23 15:00:56.056][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-23 15:00:56.056][patch_getenv.py:15] get env RES_PATH = res/20260723/qwen3_8b_qwen3_8b_a8w8_baseline_profiler
[INFO][2026-07-23 15:00:56.056][infer.py:146] Inference Configuration
[INFO][2026-07-23 15:00:56.056][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8', output_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen

---
## 6 自定义算子接入模型

### 6.1 定位替换点

Qwen3-8B 推理框架中的 W8A8 量化 matmul 通过 `CompressedTensorsW8A8Int8LinearMethod` 类实现，其 `apply_weights` 方法中使用 `torch_npu.npu_quant_matmul` 执行 INT8 量化矩阵乘法。我们需要将这个调用替换为自定义的 `torch.ops.ascendc_ops.qmm_custom` 算子。

查看原始实现：

In [9]:
!grep -n 'npu_quant_matmul' {RECIPE_ROOT}/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py

93:        # --- 替换核心: npu_quant_matmul -> qmm_custom ---
94:        # --- 替换核心: npu_quant_matmul → qmm_custom ---
95:        # --- 替换核心: npu_quant_matmul → qmm_custom ---
96:        # --- 替换核心: npu_quant_matmul → qmm_custom ---
97:        # --- 替换核心: npu_quant_matmul → qmm_custom ---
98:        x = torch_npu.npu_quant_matmul(x, layer.weight,


### 6.2 替换实现

在 `CompressedTensorsW8A8Int8LinearMethod` 中，将 `torch_npu.npu_quant_matmul` 替换为 `torch.ops.ascendc_ops.qmm_custom`。

替换后的方法为：

```python
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

def apply_weights(self, layer: torch.nn.Module,
                x: torch.Tensor,
                bias: Optional[torch.Tensor],
                dynamic_scale: Optional = None,
                out_dtype: Optional = torch.bfloat16
                ) -> Union[torch.Tensor, Dict[str, Any]]:
    if dynamic_scale is not None:
        x_scale = dynamic_scale
    else:
        x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)
    out_shape_dim_0 = x.size()[:-1]
    x = x.view(-1, x.size(-1))
    x_scale = x_scale.view(-1)

    # --- 替换核心: npu_quant_matmul → qmm_custom ---
    x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight, 
                                layer.weight_scale.view(-1),
                                pertoken_scale=None if out_dtype == torch.int32 else x_scale)
    # --- 替换结束 ---

    x = x.view(out_shape_dim_0 + (-1, ))
    if out_dtype == torch.int32:
        return x, x_scale
    else:
        return x
```

执行如下操作替换推理网络中调用的接口实现：

In [10]:
# 6.2 替换实现: 将 torch_npu.npu_quant_matmul 替换为 torch.ops.ascendc_ops.qmm_custom
import re

W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
print('[替换] 目标文件:', W8A8_FILE)

code = W8A8_FILE.read_text(encoding='utf-8')
_lib_path = str(QMM_CUSTOM_DIR / "build/libascendc_ops.so")
_load_lib_stmt = 'torch.ops.load_library(r"' + _lib_path + '")'

# 1: 清理所有旧 load_library（兼容各种注释格式），写入新的
code = re.sub(
    r'^[ \t]*(?:#\s*(?:\[临时注释-基线测试\]\s*)*)*torch\.ops\.load_library\([^)]+\)[ \t]*(?:#.*)?\n',
    '',
    code,
    flags=re.MULTILINE
)
code = code.replace('import torch\n', 'import torch\n' + _load_lib_stmt + '\n')
print('[插入] torch.ops.load_library 已写入')

# 2: 替换 npu_quant_matmul → qmm_custom
OLD_NPU = (
    'x = torch_npu.npu_quant_matmul(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    bias=layer.bias,\n'
    '                                    output_dtype=out_dtype)'
)
NEW_QMM = (
    '# --- 替换核心: npu_quant_matmul → qmm_custom ---\n'
    '        x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale)\n'
    '        if layer.bias is not None:\n'
    '            x = x + layer.bias\n'
    '        # --- 替换结束 ---'
)
if OLD_NPU in code:
    # Exact match
    code = code.replace(OLD_NPU, NEW_QMM)
    W8A8_FILE.write_text(code, encoding='utf-8')
    print('[替换] npu_quant_matmul → qmm_custom (exact match)')
elif 'qmm_custom' in code:
    print('[替换] 文件中已有 qmm_custom, 无需替换')
else:
    # Regex fallback
    OLD_NPU_REGEX = (
        r'x = torch_npu\.npu_quant_matmul\(x, layer\.weight,\s*\n'
        r'\s*layer\.weight_scale\.view\(-1\),\s*\n'
        r'\s*pertoken_scale=None if out_dtype == torch\.int32 else x_scale,\s*\n'
        r'\s*bias=layer\.bias,\s*\n'
        r'\s*output_dtype=out_dtype\)'
    )
    if re.search(OLD_NPU_REGEX, code):
        code = re.sub(OLD_NPU_REGEX, NEW_QMM, code)
        W8A8_FILE.write_text(code, encoding='utf-8')
        print('[替换] npu_quant_matmul → qmm_custom (regex fallback)')
    else:
        W8A8_FILE.write_text(code, encoding='utf-8')
        print('[替换] 警告: 文件中既无 qmm_custom 也无 npu_quant_matmul, 请检查')

# 验证
result = W8A8_FILE.read_text(encoding='utf-8')
assert 'qmm_custom' in result, '替换失败: qmm_custom 未找到'
assert 'npu_quant_matmul(x, layer.weight' not in result, '替换失败: npu_quant_matmul 仍存在'
print('[验证] 替换生效, 当前使用 qmm_custom')


[替换] 目标文件: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py
[插入] torch.ops.load_library 已写入
[替换] npu_quant_matmul → qmm_custom (exact match)
[验证] 替换生效, 当前使用 qmm_custom


### 6.3 替换实现后执行量化推理

指定本地的量化模型权重路径，调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，对比替换前的生成文本是否一致。

**注意：需要先完成第五章的量化模型推理过程，并将`W8A8_WEIGHT`指定为本地的量化模型权重的路径**

In [11]:
import sys, subprocess, os
from inference_scripts.recipe_workflow import prepare_runtime_yaml

# 将W8A8_WEIGHT设置为量化模型权重实际所在路径
W8A8_WEIGHT = TUTORIAL_DIR / 'src/data/models/Qwen3-8B-W8A8'

YAML_A8W8_TEMPLATE =  RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_1tp.yaml'
YAML_A8W8_CUSTOM  = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom.yaml'
prepare_runtime_yaml(YAML_A8W8_TEMPLATE, YAML_A8W8_CUSTOM, model_path=str(W8A8_WEIGHT))

env = os.environ.copy()
python_dir = os.path.dirname(sys.executable)
env["PATH"] = python_dir + ":" + env.get("PATH", "")
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom.yaml)
world_size = 1
[INFO][2026-07-23 15:02:03.932][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-23 15:02:03.933][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-23 15:02:03.933][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-23 15:02:03.935][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-23 15:02:03.935][patch_getenv.py:15] get env RES_PATH = res/20260723/qwen3_8b_qwen3_8b_a8w8_custom
[INFO][2026-07-23 15:02:03.935][infer.py:146] Inference Configuration
[INFO][2026-07-23 15:02:03.935][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitCode/cann/cann-lear

---
## 7 模型性能测试

### 任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程

将 QmmCustom 算子接入模型后，使用 Profiling 工具观测模型推理性能，对比自定义算子替换前后的 matmul 耗时变化。

### 7.1 准备 Profiling YAML

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将 YAML 配置中的enable_profiler配置为 True ：

In [12]:
from inference_scripts.recipe_workflow import prepare_profiling_yaml

YAML_A8W8_CUSTOM_PROFILER = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml'

prepare_profiling_yaml(YAML_A8W8_CUSTOM, YAML_A8W8_CUSTOM_PROFILER, enable_profiler=True)

{'model_config': {'model_name': 'qwen3_8b',
  'model_path': '/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8',
  'exe_mode': 'eager',
  'enable_profiler': True,
  'with_ckpt': True,
  'enable_cache_compile': False,
  'enable_static_kernel': False},
 'data_config': {'dataset': 'default', 'input_truncated_len': 4096},
 'parallel_config': {'world_size': 1,
  'attn_tp_size': 1,
  'moe_tp_size': 1,
  'embed_tp_size': 1,
  'lmhead_tp_size': 1},
 'scheduler_config': {'max_new_tokens': 512, 'batch_size': 1}}

### 7.2 启动 Profiling 推理

使用 recipes 命令启动带 Profiler 的推理，推理结束后会在结果目录中生成 `prof/` 产物。

In [13]:
import shutil
# Clear old custom profiler results
_custom_res = RECIPE_ROOT / "models/qwen/res"
for _d in _custom_res.iterdir() if _custom_res.exists() else []:
    for _c in _d.iterdir():
        if "custom_profiler" in _c.name:
            shutil.rmtree(str(_c), ignore_errors=True)
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM_PROFILER],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml)
world_size = 1
[INFO][2026-07-23 15:04:34.348][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-23 15:04:34.348][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-23 15:04:34.348][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-23 15:04:34.350][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-23 15:04:34.350][patch_getenv.py:15] get env RES_PATH = res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler
[INFO][2026-07-23 15:04:34.350][infer.py:146] Inference Configuration
[INFO][2026-07-23 15:04:34.350][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitC

### 7.3 查看 Profiling 结果

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv`中展示了模型调用的所有算子的统计信息，可快速确认自定义算子 QmmCustom 的调用情况。`kernel_details.csv`中展示了详细的接口调用信息，可自行查看筛选自己关注的算子的性能。

通过如下操作统计 `kernel_details.csv` 中的 QmmCustom 算子耗时，可自行对比自定义算子替换前后的性能差异。

In [14]:
import pandas as pd
from pathlib import Path

RES_DIR = RECIPE_ROOT / 'models/qwen/res'

# 取最新的 profiling 结果目录
result_dirs = sorted([d for d in RES_DIR.iterdir() if d.is_dir()], key=lambda d: d.name, reverse=True)
latest_result = result_dirs[0]
print(f'最新结果目录: {latest_result}')

def find_kernel_csv(phase_dir):
    matches = sorted(phase_dir.rglob('kernel_details.csv'), key=lambda p: p.parent.parent.name, reverse=True)
    return matches[0] if matches else None

prefill_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill')
decode_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode')
assert prefill_csv and decode_csv, 'kernel_details.csv 未找到，请确认 profiling 已完成'
print(f'Prefill CSV: {prefill_csv}')
print(f'Decode CSV: {decode_csv}')

# 读取 CSV 并筛选 QmmCustom 算子
prefill_df = pd.read_csv(prefill_csv)
decode_df = pd.read_csv(decode_csv)

prefill_qmm = prefill_df[prefill_df['Type'].str.contains('QmmCustom')].copy()
decode_qmm = decode_df[decode_df['Type'].str.contains('QmmCustom')].copy()

# 按 Input Shapes 分组，统计每种 shape 下的平均 Duration
def qmm_duration_stats(df, phase):
    stats = df.groupby(['Input Shapes', 'Output Data Types'])['Duration(us)'].agg(['mean', 'count']).reset_index()
    stats.columns = ['Input Shapes', 'Output Dtype', 'Avg Duration(us)', 'Calls']
    stats = stats.sort_values('Avg Duration(us)', ascending=False).reset_index(drop=True)
    stats.insert(0, 'Phase', phase)
    return stats

prefill_stats = qmm_duration_stats(prefill_qmm, 'Prefill')
decode_stats = qmm_duration_stats(decode_qmm, 'Decode')

combined = pd.concat([prefill_stats, decode_stats], ignore_index=True)
display(combined.style.set_caption('QmmCustom 各 Shape 平均耗时 (us)'))

最新结果目录: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723
Prefill CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill/551ed93c092344f982abd0cb61a6644a_157738_20260723150555813_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
Decode CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode/551ed93c092344f982abd0cb61a6644a_157738_20260723150601558_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv


,Phase,Input Shapes,Output Dtype,Avg Duration(us),Calls
0,Prefill,"""50,4096;4096,6144;6144;50""",DT_BF16,7881.550556,36
1,Prefill,"""50,12288;12288,4096;4096;50""",DT_BF16,5721.757222,36
2,Prefill,"""50,4096;4096,4096;4096;50""",DT_BF16,5471.450556,36
3,Prefill,"""50,4096;4096,24576;24576""",INT32,1119.161111,36
4,Decode,"""1,4096;4096,24576;24576""",INT32,1051.605926,108
5,Decode,"""1,12288;12288,4096;4096;1""",DT_BF16,651.997037,108
6,Decode,"""1,4096;4096,6144;6144;1""",DT_BF16,421.415556,108
7,Decode,"""1,4096;4096,4096;4096;1""",DT_BF16,285.895741,108


### 7.4 对比替换前后性能

下面汇总 **替换前（`npu_quant_matmul`）** 与 **替换后（`qmm_custom`）** 的模型级 matmul 算子耗时，直观对比替换效果。

同时从 Section 5.2 和 Section 5.3 的单算子 Profiling 数据中读取各规格耗时，进行单算子级对比。

> **清理说明**：对比完成后，本 Cell 末尾会自动将模型代码恢复为默认的 `npu_quant_matmul`，并移除 `load_library` 调用，确保 Notebook 运行完毕后环境恢复干净。

In [15]:
import pandas as pd
from pathlib import Path
import csv

RES_DIR = RECIPE_ROOT / 'models/qwen/res'

# ============================================================
# Part A: 模型级对比 — 从 kernel_details.csv 统计
# ============================================================

def find_kernel_csv(phase_dir):
    """在 phase_dir 中递归查找 kernel_details.csv"""
    matches = sorted(phase_dir.rglob('kernel_details.csv'), key=lambda p: p.parent.parent.name, reverse=True)
    return matches[0] if matches else None

def collect_op_stats(csv_path, search_patterns):
    """
    从 kernel_details.csv 中筛选匹配的算子, 返回 DataFrame.
    search_patterns: 用于匹配 Type/Name 列的关键词列表
    """
    df = pd.read_csv(csv_path)
    mask = pd.Series(False, index=df.index)
    for pat in search_patterns:
        if 'Type' in df.columns:
            mask |= df['Type'].str.contains(pat, na=False)
        if 'Name' in df.columns:
            mask |= df['Name'].str.contains(pat, na=False)
    return df[mask].copy()

def op_duration_stats(df, phase, op_label):
    """按 Input Shapes 分组统计平均耗时"""
    if df.empty:
        return pd.DataFrame()
    stats = df.groupby(['Input Shapes', 'Output Data Types'])['Duration(us)'].agg(['mean', 'count']).reset_index()
    stats.columns = ['Input Shapes', 'Output Dtype', 'Avg Duration(us)', 'Calls']
    stats = stats.sort_values('Avg Duration(us)', ascending=False).reset_index(drop=True)
    stats.insert(0, 'Phase', phase)
    stats.insert(1, 'Operator', op_label)
    return stats


# -------- 定位替换前的基线 profiling 结果 --------
# 基线结果来自 5.4 节, 配置名为 *_baseline_profiler
baseline_result_dirs = sorted(
    [d for d in RES_DIR.iterdir() if d.is_dir()],
    key=lambda d: d.name, reverse=True
)

baseline_prefill_csv = None
baseline_decode_csv = None
for d in baseline_result_dirs:
    # baseline 的配置名包含 'baseline_profiler'
    for config_dir in d.iterdir():
        if 'baseline_profiler' in config_dir.name:
            baseline_prefill_csv = find_kernel_csv(config_dir / 'prof/prefill')
            baseline_decode_csv = find_kernel_csv(config_dir / 'prof/decode')
            if baseline_prefill_csv and baseline_decode_csv:
                break
    if baseline_prefill_csv and baseline_decode_csv:
        break

# -------- 定位替换后的 profiling 结果 (来自 Section 7.3) --------
# 即 Cell 26 已经找到的那个最新 custom_profiler 结果
latest_result = sorted(
    [d for d in RES_DIR.iterdir() if d.is_dir()],
    key=lambda d: d.name, reverse=True
)[0]
custom_prefill_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill')
custom_decode_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode')

print("=" * 70)
print("模型级性能对比: 替换前 (npu_quant_matmul) vs 替换后 (QmmCustom)")
print("=" * 70)

# 收集所有统计数据
all_stats = []

# --- 替换前 (npu_quant_matmul) ---
if baseline_prefill_csv and baseline_decode_csv:
    print(f'[基线] Prefill CSV: {baseline_prefill_csv}')
    print(f'[基线] Decode CSV: {baseline_decode_csv}')

    # npu_quant_matmul 在 profiling 中可能以不同名称出现
    NPU_PATTERNS = ['QuantMatmul', 'QuantBatchMatmul', 'quant_matmul']

    baseline_prefill = collect_op_stats(baseline_prefill_csv, NPU_PATTERNS)
    baseline_decode = collect_op_stats(baseline_decode_csv, NPU_PATTERNS)

    if not baseline_prefill.empty or not baseline_decode.empty:
        all_stats.append(op_duration_stats(baseline_prefill, 'Prefill', 'npu_quant_matmul (替换前)'))
        all_stats.append(op_duration_stats(baseline_decode, 'Decode', 'npu_quant_matmul (替换前)'))
    else:
        print('[基线] 警告: 未在基线 CSV 中找到 QuantMatmul 数据, 请手动检查 kernel_details.csv 中的算子名称')
else:
    print('[基线] 未找到基线 profiling 结果, 请确认 Section 5.4 已成功运行')

# --- 替换后 (qmm_custom) ---
if custom_prefill_csv and custom_decode_csv:
    print(f'[自定义] Prefill CSV: {custom_prefill_csv}')
    print(f'[自定义] Decode CSV: {custom_decode_csv}')

    CUSTOM_PATTERNS = ['QmmCustom', 'qmm_custom']

    custom_prefill = collect_op_stats(custom_prefill_csv, CUSTOM_PATTERNS)
    custom_decode = collect_op_stats(custom_decode_csv, CUSTOM_PATTERNS)

    all_stats.append(op_duration_stats(custom_prefill, 'Prefill', 'qmm_custom (替换后)'))
    all_stats.append(op_duration_stats(custom_decode, 'Decode', 'qmm_custom (替换后)'))
else:
    print('[自定义] 未找到自定义算子 profiling 结果, 请确认 Section 7.2-7.3 已成功运行')

# 合并显示
if all_stats:
    combined = pd.concat([s for s in all_stats if not s.empty], ignore_index=True)
    if not combined.empty:
        display(combined.style.set_caption('模型级 Matmul 算子耗时对比 (us)'))
    else:
        print('无数据可显示')
else:
    print('未能收集到任何 profiling 数据')

# ============================================================
# Part B: 单算子级对比
# ============================================================
print()
print("=" * 70)
print("单算子级性能对比: npu_quant_matmul vs QmmCustom")
print("=" * 70)

# 复用 Section 5.2 和 Section 5.3 中已定义的解析函数和目录路径
# 注意: 这两个变量在各自的 Cell 中定义, 在 Jupyter 中跨 Cell 可用
SINGLE_OP_CUSTOM_DIR = str(SRC_DIR / "prof_output")           # 5.2 的输出
SINGLE_OP_BASELINE_DIR = str(SRC_DIR / "prof_output_baseline")  # 5.3 的输出

def read_single_op_csv(prof_dir, search_patterns, op_label):
    """读取单算子 profiler 输出的 kernel_details.csv"""
    csv_path = None
    prof_path = Path(prof_dir)
    if prof_path.exists():
        for subdir in sorted(prof_path.iterdir(), key=lambda d: d.name, reverse=True):
            c = subdir / "ASCEND_PROFILER_OUTPUT" / "kernel_details.csv"
            if c.exists():
                csv_path = str(c)
                break

    if not csv_path:
        print(f'  [{op_label}] 未在 {prof_dir} 中找到 kernel_details.csv')
        return {}

    rows = []
    with open(csv_path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row.get("Name", "")
            type_name = row.get("Type", "")
            matched = False
            for pat in search_patterns:
                if pat in name or pat in type_name:
                    matched = True
                    break
            if not matched:
                continue

            # Parse M, N from Output Shapes (reliable for both npu and custom ops)
            out_shapes = row.get("Output Shapes", "").strip('"')
            out_parts = [p.strip() for p in out_shapes.split(",")]
            if len(out_parts) < 2:
                continue
            M = int(out_parts[0])
            N = int(out_parts[1])
            # K from Input Shapes first part
            in_shapes = row.get("Input Shapes", "").strip('"')
            in_parts = [p.strip() for p in in_shapes.split(";")]
            K = 0
            if len(in_parts) >= 1:
                x1 = [int(x) for x in in_parts[0].split(",")]
                if len(x1) >= 2:
                    K = x1[1]

            out_dtype_raw = row.get("Output Data Types", "").strip('"')
            dtype_tag = "INT32" if "INT32" in out_dtype_raw else "BF16"

            duration_us = float(row.get("Duration(us)", "0") or "0")
            rows.append({
                "M": M, "K": K, "N": N,
                "dtype": dtype_tag,
                "duration_us": duration_us,
            })

    shape_map = {}
    for kr in rows:
        key = (kr["M"], kr["K"], kr["N"])
        shape_map.setdefault(key, {})[kr["dtype"]] = kr["duration_us"]
    return shape_map

# 读取两边的单算子数据
baseline_shapes = read_single_op_csv(
    SINGLE_OP_BASELINE_DIR,
    ['QuantMatmul', 'QuantBatchMatmul', 'quant_matmul'],
    'npu_quant_matmul'
)
custom_shapes = read_single_op_csv(
    SINGLE_OP_CUSTOM_DIR,
    ['QmmCustom', 'qmm_custom'],
    'qmm_custom'
)

# 需要从 Cell 12 获取 TEST_SHAPES (已在上方 Cell 中定义, 跨 Cell 可用)
try:
    _test_shapes = TEST_SHAPES
except NameError:
    _test_shapes = [
        (1,4096,4096), (1,4096,6144), (1,4096,24576), (1,12288,4096),
        (50,4096,4096), (50,4096,6144), (50,4096,24576), (50,12288,4096),
        (4096,4096,4096), (4096,4096,6144), (4096,4096,24576), (4096,12288,4096),
    ]

# 打印对比表
print()
header = f"{'规格 (M,K,N)':>24} | {'基线 INT32(us)':>15} | {'自定义 INT32(us)':>16} | {'加速比':>8} || {'基线 BF16(us)':>15} | {'自定义 BF16(us)':>16} | {'加速比':>8}"
print(header)
print("-" * len(header))

for M, K, N in _test_shapes:
    key = (M, K, N)
    b_int32 = baseline_shapes.get(key, {}).get("INT32", None)
    c_int32 = custom_shapes.get(key, {}).get("INT32", None)
    b_bf16 = baseline_shapes.get(key, {}).get("BF16", None)
    c_bf16 = custom_shapes.get(key, {}).get("BF16", None)

    # INT32 加速比 (基线/自定义, >1 表示自定义更快)
    if isinstance(b_int32, float) and isinstance(c_int32, float) and c_int32 > 0:
        speedup_i32 = f"{b_int32/c_int32:.2f}x"
    else:
        speedup_i32 = "N/A"

    # BF16 加速比
    if isinstance(b_bf16, float) and isinstance(c_bf16, float) and c_bf16 > 0:
        speedup_bf16 = f"{b_bf16/c_bf16:.2f}x"
    else:
        speedup_bf16 = "N/A"

    b_i32_str = f"{b_int32:.1f}" if isinstance(b_int32, float) else str(b_int32 or '-')
    c_i32_str = f"{c_int32:.1f}" if isinstance(c_int32, float) else str(c_int32 or '-')
    b_bf_str = f"{b_bf16:.1f}" if isinstance(b_bf16, float) else str(b_bf16 or '-')
    c_bf_str = f"{c_bf16:.1f}" if isinstance(c_bf16, float) else str(c_bf16 or '-')

    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {b_i32_str:>15} | {c_i32_str:>16} | {speedup_i32:>8} || {b_bf_str:>15} | {c_bf_str:>16} | {speedup_bf16:>8}")

print()
print("加速比 = 基线耗时 / 自定义耗时, >1 表示自定义算子更快")
print("=" * 70)


# ============================================================
# Part C: 恢复模型代码为原始 npu_quant_matmul
# ============================================================
print()
print("=" * 60)
print("[清理] 恢复模型代码为默认的 npu_quant_matmul...")
print("=" * 60)

import re
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
restored_code = W8A8_FILE.read_text(encoding='utf-8')

# 目标: 恢复为 npu_quant_matmul, 移除 load_library

# 1. 替换任何 qmm_custom 调用为 npu_quant_matmul
OLD_QMM_PATTERN = (
    r'x = torch\.ops\.ascendc_ops\.qmm_custom\(x, layer\.weight,\s*\n'
    r'\s*layer\.weight_scale\.view\(-1\),\s*\n'
    r'\s*pertoken_scale=None if out_dtype == torch\.int32 else x_scale\)'
)
OLD_NPU_CALL = (
    'x = torch_npu.npu_quant_matmul(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    bias=layer.bias,\n'
    '                                    output_dtype=out_dtype)'
)
_restored = False
if re.search(OLD_QMM_PATTERN, restored_code):
    restored_code = re.sub(OLD_QMM_PATTERN, OLD_NPU_CALL, restored_code)
    _restored = True
    print('[清理] qmm_custom → npu_quant_matmul')

# 2. 移除所有形式的 load_library 行
restored_code = re.sub(
    r'^[ \t]*(?:#\s*(?:\[临时注释-基线测试\]\s*)*)*torch\.ops\.load_library\([^)]+\)[ \t]*(?:#.*)?\n',
    '',
    restored_code,
    flags=re.MULTILINE
)
if 'torch.ops.load_library' not in restored_code and _restored:
    print('[清理] load_library 已移除')

if _restored:
    W8A8_FILE.write_text(restored_code, encoding='utf-8')
    print('[清理] 已恢复为 npu_quant_matmul, 移除了 load_library 和 qmm_custom 调用')
else:
    print('[清理] 模型代码已处于原始状态 (npu_quant_matmul), 无需恢复')

print('[清理] 完成 — Notebook 运行完毕, 模型代码已还原为默认状态')
print("=" * 60)

模型级性能对比: 替换前 (npu_quant_matmul) vs 替换后 (QmmCustom)
[基线] Prefill CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_baseline_profiler/prof/prefill/551ed93c092344f982abd0cb61a6644a_153992_20260723150121453_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
[基线] Decode CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_baseline_profiler/prof/decode/551ed93c092344f982abd0cb61a6644a_153992_20260723150126236_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
[自定义] Prefill CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill/551ed93c092344f982abd0cb61a6644a_157738_20260723150555813_ascend_pt/ASCEND_PROFILER_OUTPUT

,Phase,Operator,Input Shapes,Output Dtype,Avg Duration(us),Calls
0,Prefill,npu_quant_matmul (替换前),"""50,4096;768,256,16,32;24576""",INT32,76.266111,36
1,Prefill,npu_quant_matmul (替换前),"""50,12288;128,768,16,32;4096;50""",DT_BF16,46.835000,36
2,Prefill,npu_quant_matmul (替换前),"""50,4096;192,256,16,32;6144;50""",DT_BF16,32.263889,36
3,Prefill,npu_quant_matmul (替换前),"""50,4096;128,256,16,32;4096;50""",DT_BF16,23.773889,36
4,Decode,npu_quant_matmul (替换前),"""1,4096;768,256,16,32;24576""",INT32,76.121296,108
5,Decode,npu_quant_matmul (替换前),"""1,12288;128,768,16,32;4096;1""",DT_BF16,44.638704,108
6,Decode,npu_quant_matmul (替换前),"""1,4096;192,256,16,32;6144;1""",DT_BF16,32.843704,108
7,Decode,npu_quant_matmul (替换前),"""1,4096;128,256,16,32;4096;1""",DT_BF16,21.460741,108
8,Prefill,qmm_custom (替换后),"""50,4096;4096,6144;6144;50""",DT_BF16,7881.550556,36
9,Prefill,qmm_custom (替换后),"""50,12288;12288,4096;4096;50""",DT_BF16,5721.757222,36



单算子级性能对比: npu_quant_matmul vs QmmCustom

              规格 (M,K,N) |    基线 INT32(us) |    自定义 INT32(us) |      加速比 ||     基线 BF16(us) |     自定义 BF16(us) |      加速比
-------------------------------------------------------------------------------------------------------------------------
       M=1,K=4096,N=4096 |            17.8 |            135.8 |    0.13x ||            19.5 |            137.5 |    0.14x
       M=1,K=4096,N=6144 |            18.3 |            203.0 |    0.09x ||            24.0 |            204.8 |    0.12x
      M=1,K=4096,N=24576 |            48.5 |            902.2 |    0.05x ||            52.8 |            905.2 |    0.06x
      M=1,K=12288,N=4096 |            28.0 |            398.6 |    0.07x ||            28.4 |            401.0 |    0.07x
      M=50,K=4096,N=4096 |            14.2 |            144.7 |    0.10x ||            19.1 |            157.9 |    0.12x
      M=50,K=4096,N=6144 |            20.3 |            214.0 |    0.09x ||            23.8 |           

## 8 提交要求

> 完整的提交说明参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

1. 提交完整的 Jupyter Notebook（.ipynb），**保留所有单元格输出**。
2. 提交前依次运行所有单元格，确保：
   - 自定义算子编译成功；
   - 算子功能测试通过；
   - 性能数据收集成功。
3. **文件命名**：`gitcode账号_result.ipynb`。
4. **允许并且鼓励**使用 AI 工具辅助，但须独立思考并在报告中简述（解决了什么疑惑、是否引入新 bug）。
5. **不得抄袭。**

---
### 小结

本章完成了 QmmCustom 算子的开发、编译和单算子功能性能测试，并将该自定义算子接入了 Qwen3-8B 量化模型，验证功能并观测了其在模型中的性能。


### 附录
实践相关参考材料链接：
- [Ascend C 高阶api文档](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/%E9%AB%98%E9%98%B6API/%E9%AB%98%E9%98%B6API.md)
- [昇腾社区文档](https://www.hiascend.com/zh/document)
- [Ascend C 算子开发系列教程](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development)
- [Ascend C 算子功能调试](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development/07_Troubleshooting)
- [Matmul最佳实践样例](https://gitcode.com/cann/asc-devkit/blob/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance/README.md)
- [CANN内置量化矩阵乘算子实现](https://gitcode.com/cann/ops-nn/tree/master/matmul/quant_batch_matmul_v3)